# Agentic Pydantic

In [1]:
!pip install pydantic
!pip install pydantic-ai
!pip install nest-asyncio
!pip install devtools
!pip install logfire

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.4/211.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.3/289.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.1/374.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Pydantic Use Case

In [2]:
from pydantic import BaseModel, PositiveFloat

class ProductQuery(BaseModel):
    product_name: str
    max_price: PositiveFloat

# ข้อมูลถูกต้องตามเงื่อนไขที่กำหนด
query = ProductQuery(
    product_name="กาแฟสด",
    max_price=1000.0
)

print(query)

product_name='กาแฟสด' max_price=1000.0


## Gemini Model

In [5]:
import os
from google.colab import userdata
gemini_key = userdata.get('GEMINI_API_KEY')

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

import nest_asyncio :
By design asyncio does not allow its event loop to be nested. This presents a practical problem: When in an environment where the event loop is already running it's impossible to run tasks and wait for the result. Trying to do so will give the error "RuntimeError: This event loop is already running".

https://medium.com/@sakthiveltvt.thangaraj/introduction-to-nest-asyncio-for-python-developers-afd7bed44768

In [6]:
import nest_asyncio
nest_asyncio.apply()

Pydantic Agentic AI : https://ai.pydantic.dev/agents/#introduction

In [7]:
from pydantic_ai import Agent, ModelRetry, RunContext

agent = Agent(
  'google-gla:gemini-2.0-flash',
  system_prompt='Be concise, reply with one sentence.',
)

result = agent.run_sync('ช่วยแนะนำอาหารไทยเผ็ดให้หน่อย?')
print(result.data)

ผัดกะเพราเป็นอาหารไทยรสจัดจ้านที่ทำจากเนื้อสัตว์หรือเต้าหู้ผัดกับใบกะเพรา พริก และกระเทียม


/tmp/ipython-input-7-1687363187.py:9: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(result.data)


## Add Geo Tools

In [9]:
import os
from google.colab import userdata

os.environ["WEATHER_API_KEY"] = userdata.get('WEATHER_API_KEY').strip()
os.environ["GEO_API_KEY"] = userdata.get('GEO_API_KEY').strip()

from dataclasses import dataclass : This module provides a decorator and functions for automatically adding generated special methods such as __init__() and __repr__() to user-defined classes. [https://docs.python.org/3/library/dataclasses.html]

In [10]:
from __future__ import annotations as _annotations
import asyncio
import os
from dataclasses import dataclass
from typing import Any
import logfire
from devtools import debug
from httpx import AsyncClient

In [13]:
logfire.configure(send_to_logfire='if-token-present')


@dataclass
class Deps:
    client: AsyncClient
    weather_api_key: str | None
    geo_api_key: str | None


weather_agent = Agent(
    'google-gla:gemini-2.0-flash',
    system_prompt='Be concise, reply with one sentence in Thai.',
    deps_type=Deps,
    retries=2,
)


@weather_agent.tool
async def get_lat_lng(
    ctx: RunContext[Deps], location_description: str
) -> dict[str, float]:
    """Get the latitude and longitude of a location.

    Args:
        ctx: The context.
        location_description: A description of a location.
    """
    if ctx.deps.geo_api_key is None:
        return {'lat': 51.1, 'lng': -0.1}

    params = {
        'q': location_description,
        'api_key': ctx.deps.geo_api_key,
    }
    with logfire.span('calling geocode API', params=params) as span:
        r = await ctx.deps.client.get('https://geocode.maps.co/search', params=params)
        r.raise_for_status()
        data = r.json()
        span.set_attribute('response', data)

    if data:
        return {'lat': data[0]['lat'], 'lng': data[0]['lon']}
    else:
        raise ModelRetry('Could not find the location')



@weather_agent.tool
async def get_weather(ctx: RunContext[Deps], lat: float, lng: float) -> dict[str, Any]:
    """Get the weather at a location.

    Args:
        ctx: The context.
        lat: Latitude of the location.
        lng: Longitude of the location.
    """
    if ctx.deps.weather_api_key is None:
        # if no API key is provided, return a dummy response
        return {'temperature': '21 °C', 'description': 'Sunny'}

    params = {
        'apikey': ctx.deps.weather_api_key,
        'location': f'{lat},{lng}',
        'units': 'metric',
    }
    with logfire.span('calling weather API', params=params) as span:
        r = await ctx.deps.client.get(
            'https://api.tomorrow.io/v4/weather/realtime', params=params
        )
        r.raise_for_status()
        data = r.json()
        span.set_attribute('response', data)

    values = data['data']['values']
    # https://docs.tomorrow.io/reference/data-layers-weather-codes
    code_lookup = {
        1000: 'Clear, Sunny',
        1100: 'Mostly Clear',
        1101: 'Partly Cloudy',
        1102: 'Mostly Cloudy',
        1001: 'Cloudy',
        2000: 'Fog',
        2100: 'Light Fog',
        4000: 'Drizzle',
        4001: 'Rain',
        4200: 'Light Rain',
        4201: 'Heavy Rain',
        5000: 'Snow',
        5001: 'Flurries',
        5100: 'Light Snow',
        5101: 'Heavy Snow',
        6000: 'Freezing Drizzle',
        6001: 'Freezing Rain',
        6200: 'Light Freezing Rain',
        6201: 'Heavy Freezing Rain',
        7000: 'Ice Pellets',
        7101: 'Heavy Ice Pellets',
        7102: 'Light Ice Pellets',
        8000: 'Thunderstorm',
    }
    return {
        'temperature': f'{values["temperatureApparent"]:0.0f}°C',
        'description': code_lookup.get(values['weatherCode'], 'Unknown'),
    }



async def main():
    async with AsyncClient() as client:
        # create a free API key at https://www.tomorrow.io/weather-api/
        weather_api_key = os.getenv('WEATHER_API_KEY')
        # create a free API key at https://geocode.maps.co/
        geo_api_key = os.getenv('GEO_API_KEY')
        deps = Deps(
            client=client, weather_api_key=weather_api_key, geo_api_key=geo_api_key
        )
        result = await weather_agent.run(
            'What is the weather like in Bangkok and in New Jersey?', deps=deps
        )
        debug(result)
        print('Response:', result.data)


In [14]:
asyncio.run(main())

16:10:16.336 calling geocode API
16:10:16.340 calling geocode API
16:10:18.477 calling weather API
16:10:18.479 calling weather API
/tmp/ipython-input-13-2653790722.py:121 main
    result: AgentRunResult(
        output='อากาศที่กรุงเทพฯ มีเมฆมาก อุณหภูมิ 31°C ส่วนที่นิวเจอร์ซีย์ ท้องฟ้าแจ่มใส อุณหภูมิ 32°C ค่ะ\n',
        _output_tool_name=None,
        _state=GraphAgentState(
            message_history=[
                ModelRequest(
                    parts=[
                        SystemPromptPart(
                            content='Be concise, reply with one sentence in Thai.',
                            timestamp=datetime.datetime(2025, 6, 30, 16, 10, 14, 812294, tzinfo=datetime.timezone.utc),
                            dynamic_ref=None,
                            part_kind='system-prompt',
                        ),
                        UserPromptPart(
                            content='What is the weather like in Bangkok and in New Jersey?',
                       

/tmp/ipython-input-13-2653790722.py:122: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print('Response:', result.data)
